# Spectral elements: stiffness, convergence and mass lumping

This notebook compares conventional quadratic Lagrange elements with GLL-based spectral elements on the same 2D mesh. It also demonstrates how GLL nodal quadrature produces a diagonal mass matrix.

In [251]:
using LowLevelFEM, LinearAlgebra
using IterativeSolvers, Preconditioners
using SparseArrays

### Mesh

Generate a structured mesh of higher order quadrilateral elements. The geometry and connectivity are shared by both formulations.

In [252]:
structured_rect_mesh(n=10, order=6)

### Standard Lagrange field

Define a 2D displacement field with the conventional (equidistant-node) Lagrange basis.

In [253]:
mat = Material("body", E=2e5, ν=0.3)
U = Field([mat], type=:VectorField, dim=2, fieldName=:u, basis=:lagrange);

In [254]:
ndofs(U)

7442

### Spectral field

Define the corresponding displacement field using the GLL-based spectral basis, without changing the mesh topology.

In [255]:
mat = Material("body", E=2e5, ν=0.3)
Us = Field([mat], type=:VectorField, dim=2, fieldName=:u, basis=:spectral);

### Spectral transformation diagnostics

Before interpreting the solver results, verify that the global GLL-to-Gmsh transformation is invertible and that both formulations assemble the same matrix in the original nodal representation.

In [256]:
sem = spectralTransformation(Us; details=true)
T, R = sem.T, sem.R
Isem = spdiagm(0 => ones(Float64, size(T, 2)))

println("Spectral transformation size: ", size(T))
println("Spectral DOFs: ", sem.spectral_dofs)
println("||R*T - I||∞: ", norm(R * T - Isem, Inf))
println("Relative stiffness difference before transformation: ",
        norm(K.A - Ks.A) / norm(K.A))

Spectral transformation size: (7442, 7442)
Spectral DOFs: 7442
||R*T - I||∞: 1.987652801049696e-12
Relative stiffness difference before transformation: 0.0


### Weak-form operators

Use the same strain operator and constitutive law in both discretizations.

In [257]:
ε = SymGrad
D = ConstitutiveMatrix;

### Lagrange stiffness matrix

Assemble the plane-stress stiffness matrix with the conventional Lagrange basis.

In [258]:
K = ∫(ε(U) ⋅ D(:PlaneStress, mat) ⋅ ε(U));

### Spectral stiffness matrix

Assemble the same weak form with the spectral basis; only the interpolation changes.

In [259]:
Ks = ∫(ε(Us) ⋅ D(:PlaneStress, mat) ⋅ ε(Us));

### Boundary conditions and loading

Fix the left boundary and apply an identical horizontal traction on the right boundary.

In [260]:
bc = BoundaryCondition("left", ux=0, uy=0)

f = ∫(U ⋅ [1, 0], Γ="right")
fs = ∫(Us ⋅ [1, 0], Γ="right");

### Direct solution: Lagrange

Solve the conventional system with a direct sparse solver and record the elapsed time.

In [261]:
@time u = solveField(Symmetric(K), f, support=[bc]);

  0.702075 seconds (6.74 k allocations: 117.252 MiB, 1.17% gc time)


### Direct solution: SEM

Solve the spectral system under the same boundary conditions.

In [262]:
@time us = solveField(Symmetric(Ks), fs, support=[bc]);

  0.418134 seconds (6.27 k allocations: 738.802 MiB, 47.20% gc time)


### Direct-solver agreement

Compare the displacement degrees of freedom; the two bases span the same quadratic approximation space.

In [263]:
err = norm(DoFs(u) - DoFs(us)) / norm(DoFs(u))

4.4907471584346936e-13

### Iterative solution: Lagrange

Solve the symmetric stiffness system using preconditioned conjugate gradients (CG).

In [264]:
@time u2 = solveField(Symmetric(K), f, support=[bc], solver=:cg, preconditioner=:none, solveroptions=(reltol=1e-9, maxiter=20000));

  3.879784 seconds (16.22 k allocations: 52.141 MiB, 0.13% gc time)


### Iterative solution: SEM

Repeat the same iterative solve for the spectral discretization.

In [265]:
@time us2 = solveField(Symmetric(Ks), fs, support=[bc], solver=:cg, preconditioner=:none, solveroptions=(reltol=1e-9, maxiter=20000));

  1.290429 seconds (8.81 k allocations: 673.545 MiB, 8.17% gc time)


### CG iteration counts

`solveField` currently discards the convergence history provided by `IterativeSolvers.jl`. As a temporary diagnostic, reproduce its constrained systems and incomplete-Cholesky preconditioner, and rerun CG with `log=true` to report the actual iteration counts. **This additional cell runs both CG solves again** and is not included in the timings above.

In [266]:
# Temporary diagnostic; uses LowLevelFEM's current internal solver helpers.
function cg_iteration_count(K, rhs, support)
    prepared = LowLevelFEM.prepare_singlefield_system(K, rhs, support)
    P = LowLevelFEM._build_preconditioner(Val(:ich), prepared.A)
    _, history = cg(prepared.A, vec(prepared.B[:, 1]); Pl=P, log=true)
    return history.iters, history.isconverged
end

iters_lagrange, converged_lagrange = cg_iteration_count(Symmetric(K), f, [bc])
iters_sem, converged_sem = cg_iteration_count(Symmetric(Ks), fs, [bc])

println("Lagrange CG: $iters_lagrange iterations (converged: $converged_lagrange)")
println("SEM CG:      $iters_sem iterations (converged: $converged_sem)")

Lagrange CG: 33 iterations (converged: true)
SEM CG:      31 iterations (converged: true)


### Iterative-solver agreement

Check the relative difference between the two CG solutions.

In [267]:
err = norm(DoFs(u2) - DoFs(us2)) / norm(DoFs(u2))

1.7414820469722477e-11

### Consistent mass matrix: Lagrange

Assemble the standard displacement mass matrix. Its off-diagonal entries are generally nonzero.

In [268]:
M = ∫(U ⋅ U)
M[:,:]

7442×7442 SparseMatrixCSC{Float64, Int64} with 462722 stored entries:
⎡⡿⣯⣙⣻⠁⠀⢈⡋⠀⠀⣘⠃⠀⠀⣘⠃⠀⢀⡛⠀⠀⣘⠃⠀⢀⣘⠃⠀⢀⡛⠀⠀⣸⠷⠶⢶⣽⠶⠶⣯⎤
⎢⣷⣸⣿⣿⡛⠛⠛⣳⠶⠶⠿⠶⠶⠶⠿⠶⠤⠬⠥⠤⠤⠤⠤⠤⠤⠤⢤⣤⣤⣤⣤⣤⣤⣤⣤⣀⣀⣀⣀⣀⎥
⎢⠁⠀⣿⠈⢿⣷⣀⠉⢷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⡦⠰⢿⣠⡄⠘⠻⣦⡄⠘⠷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⢸⡇⠙⢷⣀⠉⢿⣷⣀⠉⢳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠶⠘⢻⡇⠀⠀⠙⢧⡄⠘⠻⣦⡄⠈⠳⣆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⢸⡇⠀⠀⠀⠀⠙⢶⡀⠉⢻⣶⡀⠙⢳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠶⠘⢻⡇⠀⠀⠀⠀⠀⠀⠹⢦⣄⠈⠻⣦⡄⠉⠷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⢀⡀⡇⠀⠀⠀⠀⠀⠀⠀⠀⠙⢶⡄⠉⢻⣶⡀⠙⢳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠛⠈⠁⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢧⣄⠈⠿⣧⣄⠉⠷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣀⢠⠀⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢶⡄⠙⢻⣶⡀⠘⢳⣄⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠉⠀⠀⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢧⣀⠈⠿⣧⣀⠈⢷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣀⢰⠀⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢶⡀⠘⢻⣶⡀⠘⠳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠉⠀⠀⣷⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠙⢷⣀⠈⠿⣧⣀⠉⢷⣤⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣤⠰⠀⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢦⡄⠘⠻⣦⡄⠘⠷⣄⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⣷⣀⠉⢿⣷⣀⠉⢷⣄⠀⠀⠀⠀⎥
⎢⢶⡞⠀⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢧⡄⠘⠻⣦⡄⠘⢳⣄⠀⠀⎥
⎢⢸⣇⠀⢻⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢷⣀⠉⢿⣷⣀⠙⢷⡀⎥
⎢⢳⡟⠀⢸⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢶⣄⠘⢻⣶⣀⠉⎥
⎣⡼⣧⠀⢸⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⠳⡄⠘⠻⣦⎦

### Consistent mass matrix: SEM

Using conventional quadrature, the spectral element also has a non-diagonal consistent mass matrix.

In [269]:
Ms1 = ∫(Us ⋅ Us)
Ms1[:,:]

7442×7442 SparseMatrixCSC{Float64, Int64} with 462722 stored entries:
⎡⡿⣯⣙⣻⠁⠀⢈⡋⠀⠀⣘⠃⠀⠀⣘⠃⠀⢀⡛⠀⠀⣘⠃⠀⢀⣘⠃⠀⢀⡛⠀⠀⣸⠷⠶⢶⣽⠶⠶⣯⎤
⎢⣷⣸⣿⣿⡛⠛⠛⣳⠶⠶⠿⠶⠶⠶⠿⠶⠤⠬⠥⠤⠤⠤⠤⠤⠤⠤⢤⣤⣤⣤⣤⣤⣤⣤⣤⣀⣀⣀⣀⣀⎥
⎢⠁⠀⣿⠈⢿⣷⣀⠉⢷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⡦⠰⢿⣠⡄⠘⠻⣦⡄⠘⠷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⢸⡇⠙⢷⣀⠉⢿⣷⣀⠉⢳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠶⠘⢻⡇⠀⠀⠙⢧⡄⠘⠻⣦⡄⠈⠳⣆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⢸⡇⠀⠀⠀⠀⠙⢶⡀⠉⢻⣶⡀⠙⢳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠶⠘⢻⡇⠀⠀⠀⠀⠀⠀⠹⢦⣄⠈⠻⣦⡄⠉⠷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⢀⡀⡇⠀⠀⠀⠀⠀⠀⠀⠀⠙⢶⡄⠉⢻⣶⡀⠙⢳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠛⠈⠁⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢧⣄⠈⠿⣧⣄⠉⠷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣀⢠⠀⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢶⡄⠙⢻⣶⡀⠘⢳⣄⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠉⠀⠀⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢧⣀⠈⠿⣧⣀⠈⢷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣀⢰⠀⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢶⡀⠘⢻⣶⡀⠘⠳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠉⠀⠀⣷⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠙⢷⣀⠈⠿⣧⣀⠉⢷⣤⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣤⠰⠀⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢦⡄⠘⠻⣦⡄⠘⠷⣄⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⣷⣀⠉⢿⣷⣀⠉⢷⣄⠀⠀⠀⠀⎥
⎢⢶⡞⠀⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢧⡄⠘⠻⣦⡄⠘⢳⣄⠀⠀⎥
⎢⢸⣇⠀⢻⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢷⣀⠉⢿⣷⣀⠙⢷⡀⎥
⎢⢳⡟⠀⢸⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢶⣄⠘⢻⣶⣀⠉⎥
⎣⡼⣧⠀⢸⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⠳⡄⠘⠻⣦⎦

### GLL nodal quadrature: diagonal mass matrix

When interpolation and quadrature use the same GLL nodes, the Kronecker-delta property makes the mass matrix diagonal. This is especially useful for explicit dynamics; the quadrature is not fully exact for the mass integrand.

In [270]:
Ms2 = ∫(Us ⋅ Us, gauss=:gll)
Ms2[:,:]

7442×7442 SparseMatrixCSC{Float64, Int64} with 462722 stored entries:
⎡⡿⣯⣙⣻⠁⠀⢈⡋⠀⠀⣘⠃⠀⠀⣘⠃⠀⢀⡛⠀⠀⣘⠃⠀⢀⣘⠃⠀⢀⡛⠀⠀⣸⠷⠶⢶⣽⠶⠶⣯⎤
⎢⣷⣸⣿⣿⡛⠛⠛⣳⠶⠶⠿⠶⠶⠶⠿⠶⠤⠬⠥⠤⠤⠤⠤⠤⠤⠤⢤⣤⣤⣤⣤⣤⣤⣤⣤⣀⣀⣀⣀⣀⎥
⎢⠁⠀⣿⠈⢿⣷⣀⠉⢷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⡦⠰⢿⣠⡄⠘⠻⣦⡄⠘⠷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⢸⡇⠙⢷⣀⠉⢿⣷⣀⠉⢳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠶⠘⢻⡇⠀⠀⠙⢧⡄⠘⠻⣦⡄⠈⠳⣆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⢸⡇⠀⠀⠀⠀⠙⢶⡀⠉⢻⣶⡀⠙⢳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠶⠘⢻⡇⠀⠀⠀⠀⠀⠀⠹⢦⣄⠈⠻⣦⡄⠉⠷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⢀⡀⡇⠀⠀⠀⠀⠀⠀⠀⠀⠙⢶⡄⠉⢻⣶⡀⠙⢳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠛⠈⠁⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢧⣄⠈⠿⣧⣄⠉⠷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣀⢠⠀⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢶⡄⠙⢻⣶⡀⠘⢳⣄⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠉⠀⠀⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢧⣀⠈⠿⣧⣀⠈⢷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣀⢰⠀⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢶⡀⠘⢻⣶⡀⠘⠳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠉⠀⠀⣷⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠙⢷⣀⠈⠿⣧⣀⠉⢷⣤⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣤⠰⠀⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢦⡄⠘⠻⣦⡄⠘⠷⣄⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⣷⣀⠉⢿⣷⣀⠉⢷⣄⠀⠀⠀⠀⎥
⎢⢶⡞⠀⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢧⡄⠘⠻⣦⡄⠘⢳⣄⠀⠀⎥
⎢⢸⣇⠀⢻⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢷⣀⠉⢿⣷⣀⠙⢷⡀⎥
⎢⢳⡟⠀⢸⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢶⣄⠘⢻⣶⣀⠉⎥
⎣⡼⣧⠀⢸⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⠳⡄⠘⠻⣦⎦

### Verify diagonality in spectral coordinates

Because the public `SystemMatrix` uses the original Gmsh nodal representation, test the transformed matrix $T^T M_{\mathrm{GLL}} T$. The off-diagonal norm should be close to roundoff if the transformation and quadrature are consistent.

In [271]:
M_sem = sparse(T' * Ms2.A * T)
D_sem = spdiagm(0 => diag(M_sem))
relative_offdiag = norm(M_sem - D_sem) / norm(M_sem)

println("Relative off-diagonal norm of the spectral mass matrix: ", relative_offdiag)
println("Minimum diagonal entry: ", minimum(diag(M_sem)))
println("Number of explicitly stored entries in public Gmsh matrix: ", nnz(Ms2.A))
println("Number of explicitly stored entries in spectral matrix: ", nnz(M_sem))

Relative off-diagonal norm of the spectral mass matrix: 2.872845297947592e-13
Minimum diagonal entry: 5.668934240062343e-6
Number of explicitly stored entries in public Gmsh matrix: 462722
Number of explicitly stored entries in spectral matrix: 462722


In [272]:
M_sem[1:8, 1:8]

8×8 SparseMatrixCSC{Float64, Int64} with 32 stored entries:
  5.66893e-6     ⋅           -1.34665e-17  …  -5.23319e-18    ⋅ 
   ⋅            5.66893e-6     ⋅                ⋅           -5.23319e-18
 -1.3467e-17     ⋅            3.29555e-5      -1.40979e-17    ⋅ 
   ⋅           -1.3467e-17     ⋅                ⋅           -1.40979e-17
  2.81085e-18    ⋅            1.50349e-17     -3.18558e-18    ⋅ 
   ⋅            2.81085e-18    ⋅           …    ⋅           -3.18558e-18
 -5.23275e-18    ⋅           -1.40957e-17      5.80499e-5     ⋅ 
   ⋅           -5.23275e-18    ⋅                ⋅            5.80499e-5

In [273]:
M_sem[200:208, 200:208]

9×9 SparseMatrixCSC{Float64, Int64} with 41 stored entries:
 0.000298796    ⋅            2.44284e-18  …   ⋅           6.50299e-18
  ⋅            0.00046601     ⋅              1.27701e-18   ⋅ 
 2.46275e-18    ⋅            0.00046601       ⋅           1.27701e-18
  ⋅            1.0625e-18     ⋅              1.54391e-19   ⋅ 
 3.71257e-18    ⋅            1.0625e-18       ⋅           1.54391e-19
  ⋅           -2.76959e-18    ⋅           …  7.69768e-18   ⋅ 
 5.70371e-19    ⋅           -2.76959e-18      ⋅           7.69768e-18
  ⋅            1.25784e-18    ⋅              0.000298796   ⋅ 
 6.50871e-18    ⋅            1.25784e-18      ⋅           0.000298796

In [274]:
M_sem[108:116, 1000:1008]

9×9 SparseMatrixCSC{Float64, Int64} with 9 stored entries:
   ⋅             ⋅             ⋅           …    ⋅            ⋅ 
   ⋅             ⋅             ⋅                ⋅            ⋅ 
   ⋅             ⋅             ⋅                ⋅            ⋅ 
   ⋅           -1.01718e-18    ⋅              -7.9325e-18    ⋅ 
 -8.58504e-18    ⋅           -1.01718e-18       ⋅          -7.9325e-18
   ⋅             ⋅             ⋅           …    ⋅            ⋅ 
   ⋅             ⋅             ⋅                ⋅            ⋅ 
   ⋅             ⋅             ⋅                ⋅            ⋅ 
   ⋅             ⋅             ⋅                ⋅            ⋅ 